In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''

Created on 2024-07-12
Last modified on 2024-07-12
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener información a partir de los .md generados del bloque get_cti_information. Es requerido haber ejecutado previamente el módulo get_cti_information y el módulo mitre_relationships.

'''

'\n\nCreated on 2024-07-12\nLast modified on 2024-07-12\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener información a partir de los .md generados del bloque get_cti_information. Es requerido haber ejecutado previamente el módulo get_cti_information y el módulo mitre_relationships.\n\n'

**Índide de contenidos**

[1. Búsqueda de técnica (ID y nombre)]()

- [1.1 Obtenemos tabla de información encontrada por ID]()

    - [1.1.1 Añadimos la información en caso de encontrar referencias en el documento]()

- [1.2 Obtenemos tabla de información encontrada por nombre de técnica]()

[1.2.1 Añadimos la información en caso de encontrar referencias en el documento]()



[2. Búsqueda de táctica (ID y nombre)]()

- [2.1 Obtenemos tabla de información encontrada por ID]()

    - [2.1.1 Añadimos la información en caso de encontrar referencias en el documento]()

- [2.2 Obtenemos tabla de información encontrada por nombre de táctica]()

    - [2.2.1 Añadimos la información en caso de encontrar referencias en el documento]()



[3. Búsqueda de data source (ID y nombre)]()

- [3.1 Obtenemos tabla de información encontrada por ID]()

    - [3.1.1 Añadimos la información en caso de encontrar referencias en el documento]()

- [3.2 Obtenemos tabla de información encontrada por nombre de data source]()

    - [3.2.1 Añadimos la información en caso de encontrar referencias en el documento]()



[4. Búsqueda de plataforma (nombre)]()

- [4.1 Obtenemos tabla de información encontrada por ID]()

    - [4.1.1 Añadimos la información en caso de encontrar referencias en el documento]()



[5. Búsqueda de software (ID y nombre)]()

- [5.1 Obtenemos tabla de información encontrada por ID]()

    - [5.1.1 Añadimos la información en caso de encontrar referencias en el documento]()

- [5.2 Obtenemos tabla de información encontrada por nombre de software]()

    - [5.2.1 Añadimos la información en caso de encontrar referencias en el documento]()



[6. Búsqueda de grupo (ID y nombre)]()

- [6.1 Obtenemos tabla de información encontrada por ID]()

    - [6.1.1 Añadimos la información en caso de encontrar referencias en el documento]()

- [6.2 Obtenemos tabla de información encontrada por nombre de grupo]()

    - [6.2.1 Añadimos la información en caso de encontrar referencias en el documento]()

**Requerimientos**

In [2]:
import os
import re

import pandas as pd

from stix2 import Filter, MemoryStore
import stix2
import requests

##### **Parámetros**

In [3]:
matrix = 'enterprise' # enterprise / ics / mobile
news_sources = ['thehackernews']# ['thehackernews','nist', 'cyble'] Lista de outputs a evaluar
add_information = True # Parámetro que controla el añadido de información de acuerdo a las referencias encontradas en la noticia
different_output_folder = '' #'add_information' Si queremos guardar en output en una subcarpeta distinta y no reemplazar los archivos existentes

##### **Funciones**

In [4]:
def get_data_from_branch(matrix):
    '''
    Función encargada de peticionar a la url de GitHub donde está publicada la última versión MITRE de la información en formato stix2. Retorna el objeto que contiene toda la información de MITRE.
    '''
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    stix_json = requests.get(url).json()
    return MemoryStore(stix_data=stix_json["objects"])

In [5]:
def get_list_techniques_from_stix2(src, include="both"):
    '''
    Función encargada de la obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan tanto técnicas como subtécnicas. Retorna una lista de strings con el id correspondiente a cada técnica.
    '''
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [6]:
def get_techniques_lists(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las listas de id técnicas y nombre de técnicas.
    '''
    techniques_data = []
    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    for tech in techniques:
        deprecated = False
        revoked = False
        description = ''
        if 'x_mitre_deprecated' in tech: 
            deprecated = tech['x_mitre_deprecated']
        if 'revoked' in tech: 
            revoked = tech['revoked']
        techniques_data.append({
            "technique_ID": tech['external_references'][0]['external_id'],
            "technique": tech['name'],
            "technique_deprecated": deprecated,
            "technique_revoked": revoked
        })
        
    techniques_df = pd.DataFrame(techniques_data)

    if revoked_deprecated:
        techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    
    techniques_df = techniques_df[['technique_ID','technique']]

    techniques_id_list = techniques_df['technique_ID'].tolist()
    techniques_name_list = techniques_df['technique'].tolist()

    return techniques_id_list, techniques_name_list

In [7]:
def get_tactics_lists(matrix_store):
    '''
    Función que retorna las listas de id táctica y nombre de táctica.
    '''
    tactics_data = []
    tactics = matrix_store.query([Filter('type', '=', 'x-mitre-tactic')])
    for tact in tactics:
        tactics_data.append({
            "tactic_ID": tact['external_references'][0]['external_id'],
            "tactic": tact['name']
        })

    tactics_df = pd.DataFrame(tactics_data)
    tactics_df = tactics_df[['tactic_ID','tactic']]

    tactics_id_list = tactics_df['tactic_ID'].tolist()
    tactics_name_list = tactics_df['tactic'].tolist()

    return tactics_id_list, tactics_name_list

In [8]:
def get_datasources_lists(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las listas de id data source y nombre de data source.
    '''
    datasoruces_data = []
    data_sources = matrix_store.query([Filter('type', '=', 'x-mitre-data-source')])
    for ds in data_sources:
        deprecated = False
        revoked = False
        if 'x_mitre_deprecated' in ds: 
            deprecated = ds['x_mitre_deprecated']
        if 'revoked' in ds: 
            revoked = ds['revoked']

        datasoruces_data.append({
            "datasource_ID": ds['external_references'][0]['external_id'],
            "datasource": ds['name'],
            "datasource_deprecated": deprecated,
            "datasource_revoked": revoked
        })

    datasoruces_df = pd.DataFrame(datasoruces_data)
    if revoked_deprecated:
        datasoruces_df = datasoruces_df[(datasoruces_df['datasource_deprecated']!=True)&(datasoruces_df['datasource_revoked']!=True)]

    datasoruces_df = datasoruces_df[['datasource_ID','datasource']]
    datasoruces_id_list = datasoruces_df['datasource_ID'].tolist()
    datasoruces_name_list = datasoruces_df['datasource'].tolist()
    
    return datasoruces_id_list, datasoruces_name_list

In [9]:
def get_platforms_list(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las lista de plataformas.
    '''
    platforms_from_tech = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    platforms_data = []
    for technique in platforms_from_tech:
        platform_name = 'None'
        if 'x_mitre_data_sources' in technique: 
            try: 
                platform_name = technique['x_mitre_platforms']
            except:
                platform_name = 'None'

        platforms_data.append({
            "platform": platform_name,
        })
    # Generamos df a partir de los datos recopilados
    platforms_df = pd.DataFrame(platforms_data)
    platforms_df = platforms_df.explode('platform')
    platforms_df = platforms_df.drop_duplicates()
    platforms_df = platforms_df.sort_values(by='platform')
    platforms_df = platforms_df.reset_index(drop=True)
    platforms_name_list = platforms_df['platform'].tolist()
    
    return platforms_name_list

In [10]:
def get_software_lists(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las listas de id software y nombre de software.
    '''
    malware = matrix_store.query([Filter('type', '=', 'malware')])
    tool = matrix_store.query([Filter('type', '=', 'tool')])
    software = malware + tool
    software_data = []
    for sw in software:
        deprecated = False
        revoked = False
        if 'x_mitre_deprecated' in sw: 
            deprecated = sw['x_mitre_deprecated']
        if 'revoked' in sw: 
            revoked = sw['revoked']

        software_data.append({
            "software_ID": sw['external_references'][0]['external_id'],
            "software": sw['name'],
            "software_deprecated": deprecated,
            "software_revoked": revoked
        })


    # Generamos df a partir de los datos recopilados
    software_df = pd.DataFrame(software_data)
    revoked_deprecated = True
    if revoked_deprecated:
        software_df = software_df[(software_df['software_deprecated']!=True)&(software_df['software_revoked']!=True)]
    
    software_df = software_df[['software_ID','software']]
    software_id_list = software_df['software_ID'].tolist()
    software_name_list = software_df['software'].tolist()
    
    return software_id_list, software_name_list

In [11]:
def get_groups_lists(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las listas de id grupo y nombre de grupo.
    '''
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])
    groups_data = []
    for group in groups:
        deprecated = False
        revoked = False
        if 'x_mitre_deprecated' in group: 
            deprecated = group['x_mitre_deprecated']
        if 'revoked' in group: 
            revoked = group['revoked']

        groups_data.append({
            "group_ID": group['external_references'][0]['external_id'],
            "group": group['name'],
            "group_deprecated": deprecated,
            "group_revoked": revoked
        })


    # Generamos df a partir de los datos recopilados
    groups_df = pd.DataFrame(groups_data)
    revoked_deprecated = True
    if revoked_deprecated:
        groups_df = groups_df[(groups_df['group_deprecated']!=True)&(groups_df['group_revoked']!=True)]
    
    groups_df = groups_df[['group_ID','group']]
    groups_id_list = groups_df['group_ID'].tolist()
    groups_name_list = groups_df['group'].tolist()
    
    return groups_id_list, groups_name_list

In [12]:
def get_list_of_files_sub(dir_name):
    '''
    Función encargada de retornar una lista de archivos ubicados en la ruta facilitada así como en los subdirectorios disponibles.
    '''
    listOfFile = os.listdir(dir_name)
    allFiles = list()
    for entry in listOfFile:
        fullPath = os.path.join(dir_name, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_of_files_sub(fullPath)
        else:
            allFiles.append(fullPath)
    return allFiles

In [13]:
def find_content_in_list(texto, items_list):
    '''
    Función encargada de buscar en un texto facilitado los elementos contenidos en la lista de técnicas. Retorna una lista con el contenido encontrado.
    '''
    if isinstance(texto, str):
        found = set()  # Usar un conjunto para almacenar elementos únicos
        for item in items_list:
            if item in texto:
                found.add(item.upper())  # Añadir el elemento al conjunto
        return list(found)  # Convertir el conjunto a lista antes de devolver
    else:
        return []

In [14]:
def list_to_string(cell):
    '''
    Función para convertir listas en cadenas separadas por ";"
    '''
    if isinstance(cell, list):
        return "; ".join(cell)
    return cell

In [15]:
def clean_col(cell):
    '''
    Función para eliminar caracteres no alfanuméricos excepto ".", y "," de cada celda de un dataframe de pandas.
    '''
    if isinstance(cell, str):
        cleaned_text = re.sub(r'[^a-zA-Z0-9\s.,-]', '', cell)
        return cleaned_text
    return cell

In [16]:
def clean_description_col(cell):
    '''
    Función específica de la columna descriptiva para eliminar caracteres no alfanuméricos excepto ".", y "," de cada celda.
    '''
    if isinstance(cell, str):
        # Expresión regular para encontrar URLs, no incluyendo caracteres de cierre como ). o ))
        urls = re.findall(r'https?://[^\s\)]+', cell)

        # Reemplazar URLs con marcadores {url1}, {url2}, etc.
        for i, url in enumerate(urls):
            cell = cell.replace(url, f'{{url{i}}}')
        # Eliminar caracteres no alfanuméricos excepto "/", ":", y espacios
        temp_text = re.sub(r'[^a-zA-Z0-9\s.,{}]', '', cell)

        # Reinsertar las URLs en el texto limpio
        for i, url in enumerate(urls):
            temp_text = temp_text.replace(f'{{url{i}}}', " " + url)

        # Reemplazar múltiples saltos de línea y espacios con un solo espacio
        temp_text = re.sub(r'\s+', ' ', temp_text).strip() # nuevo
        
        return temp_text
    return cell

In [17]:
def match_items(items_list, news_sources_list):
    '''
    Función encargada de devolver un diccionario compuesto por la ruta del archivo analizado como clave y como valor una lista que a su vez se compone por el numero de coincidencias encontradas y las propias coincidencias (únicas).
    '''
    items_list = [item.lower() for item in items_list]
    results = {}
    for ns in news_sources_list: # recorremos la lista de outputs (thehackernews, nist, cyble)
        review_path = os.path.join(os.path.dirname(os.getcwd()), 'get_cti_information', 'outputs', ns)
        items_md = get_list_of_files_sub(review_path)
        items_md = [item for item in items_md if os.path.splitext(item)[1] == '.md']# eliminamos posibles archivos cuya extension no sea .md 
        
        for md_file in items_md:
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                with open(md_file, 'r', encoding='utf-8') as file:
                    md_content = file.read()
                    md_content = md_content.lower()
                    md_content = re.sub(r'---.*?---', '', md_content, flags=re.DOTALL) # Eliminamos el header del contenido que vamos a revisar
                    items_found = find_content_in_list(md_content, items_list)
                    results[md_file] = [len(items_found), items_found]
            except:
                pass
    return results

In [18]:
def results_to_df(dict_results):
    '''
    Función para convertir en df el diccionario generado por match_items()
    '''
    data_tuples = [(key, *value) for key, value in dict_results.items()]
    dict_results_df = pd.DataFrame(data_tuples, columns=['file_path', 'match_count', 'matches'])
    dict_results_df = dict_results_df.sort_values('match_count', ascending=False)
    return dict_results_df

In [19]:
def unique_list(series):
    '''
    Función para devolver sólo ítems únicos a partir de una lista.
    '''
    return list(set(series))

In [20]:
def get_data_from_techniques(matrix):
    file_name = f'[MITRE]_{matrix}_techniques.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'information\techniques',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [21]:
def get_data_from_relation(matrix, source_rel, source_info):
    '''
    Función encargada de generar un df que contiene la información de la relación entre técnicas y propiedad así como la información complementaria relativa a la propiedad (data source, grupo, plataforma, software y táctica).
    '''
    mapping = {
        'datasources': 'data_source_ID',
        'groups': 'group_ID',
        'platforms': 'platform',
        'software': 'software_ID',
        'tactics': 'tactic_ID'
    }

    if source_info in mapping:
        rel_id = mapping[source_info]
        info_id = mapping[source_info]

    relation_name = f'[MITRE]_{matrix}_{source_rel}_NN.csv'
    path_relation = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, 'relations', source_rel, relation_name)
    information_name = f'[MITRE]_{matrix}_{source_info}.csv'
    path_info = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, 'information', source_info, information_name)
    relation_df = pd.read_csv(path_relation, sep=';', quotechar='"')
    information_df = pd.read_csv(path_info, sep=';', quotechar='"')

    # display(relation_df.head(2))
    # display(information_df.head(2))

    merged_df = pd.merge(relation_df, information_df, how='left', left_on=rel_id, right_on=info_id)

    if 'data_source_y' in merged_df.columns:
        merged_df.drop(columns=['data_source_y'], inplace=True)
        merged_df = merged_df.rename(columns={'data_source_x': 'data_source'})
        merged_df.drop(columns=[(rel_id[:-3]+'_revoked'),(rel_id[:-3]+'_deprecated')], inplace=True)
    elif 'group_y' in merged_df.columns:
        merged_df.drop(columns=['group_y'], inplace=True)
        merged_df = merged_df.rename(columns={'group_x': 'group'})
        merged_df.drop(columns=[(rel_id[:-3]+'_revoked'),(rel_id[:-3]+'_deprecated')], inplace=True)
    elif 'software_y' in merged_df.columns:
        merged_df.drop(columns=['software_y'], inplace=True)
        merged_df = merged_df.rename(columns={'software_x': 'software'})
        merged_df.drop(columns=[(rel_id[:-3]+'_revoked'),(rel_id[:-3]+'_deprecated')], inplace=True)
    elif 'tactic_y' in merged_df.columns:
        merged_df.drop(columns=['tactic_y'], inplace=True)
        merged_df = merged_df.rename(columns={'tactic_x': 'tactic'})

    # Seleccionamos las columnas a agregar, excluyendo las relacionadas de la propiedad
    columns_to_aggregate = [col for col in merged_df.columns if col not in [rel_id]]

    # Creamos un diccionario para agregar dinámicamente
    agg_dict = {col: unique_list for col in columns_to_aggregate}

    # Agrupamos y agrega usando el diccionario generado
    merged_df = merged_df.groupby(rel_id, as_index=False).agg(agg_dict)
    # display(merged_df)
    return merged_df

### **Ejecución principal**

In [22]:
mitre_matrix = get_data_from_branch(matrix)

In [23]:
techniques_id_list, techniques_name_list =  get_techniques_lists(mitre_matrix)

In [24]:
tactics_id_list, tactics_name_list =  get_tactics_lists(mitre_matrix)

In [25]:
datasources_id_list, datasources_name_list =  get_datasources_lists(mitre_matrix)

In [26]:
platforms_name_list =  get_platforms_list(mitre_matrix)

In [27]:
software_id_list, software_name_list =  get_software_lists(mitre_matrix)

In [28]:
groups_id_list, groups_name_list =  get_groups_lists(mitre_matrix)

### **1. Búsqueda de técnicas (ID y nombre)**

#### **1.1 Obtenemos tabla de información encontrada por ID**

In [29]:
results_techniques_id = match_items(techniques_id_list, news_sources)
md_search_techniqueid_df = results_to_df(results_techniques_id)
md_search_techniqueid_df = md_search_techniqueid_df.reset_index(drop=True)
md_search_techniqueid_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


#####  **1.1.1 Añadimos la información en caso de encontrar referencias en el documento**

In [30]:
if add_information:
    modified_files = []
    for new in range(md_search_techniqueid_df.shape[0]):
        file_path = md_search_techniqueid_df.loc[new, 'file_path']
        matches = md_search_techniqueid_df.loc[new, 'match_count']
        items = md_search_techniqueid_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_techniques(matrix)
            data_to_incoporate['technique_ID'] = data_to_incoporate['technique_ID'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo technique
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['technique_ID']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_techniqueid_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Se han modificado un total de 0 archivo/s.


#### **1.2 Obtenemos tabla de información encontrada por nombre de técnica**

In [31]:
results_techniques_name = match_items(techniques_name_list, news_sources)
md_search_technique_df = results_to_df(results_techniques_name)
md_search_technique_df = md_search_technique_df.reset_index(drop=True)
md_search_technique_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,14,"[PHISHING, VULNERABILITIES, HIDDEN WINDOW, SOF..."
1,c:\Users\jelopez\Documents\CyberProof\python\d...,13,"[PHISHING, KEYCHAIN, SOFTWARE, SERVER, MALWARE..."
2,c:\Users\jelopez\Documents\CyberProof\python\d...,12,"[SERVER, MALWARE, JAVASCRIPT, TOOL, POWERSHELL..."


#####  **1.2.1 Añadimos la información en caso de encontrar referencias en el documento**

In [32]:
if add_information:
    modified_files = []
    for new in range(md_search_technique_df.shape[0]):
        file_path = md_search_technique_df.loc[new, 'file_path']
        matches = md_search_technique_df.loc[new, 'match_count']
        items = md_search_technique_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_techniques(matrix)
            data_to_incoporate['technique'] = data_to_incoporate['technique'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo technique
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['technique']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_technique_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240717\new_malware_campaign_abusing_rdpwrapper_and_tailscale_to_target_cryptocurrency_users_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240223\uncovering_atomic_stealer_amos_strikes_and_the_rise_of_dead_cookies_restoration_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240717\investigating_the_new_jellyfish_loader_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240719\xehook_stealer_evolution_of_cinoshis_project_targeting_over_100_cryptocurrencies_and_2fa_extensions_-_cyble.md
Fichero generado y guardado correctamente c:\Use

### **2. Búsqueda de táctica (ID y nombre)**

#### **2.1 Obtenemos tabla de información encontrada por ID**

In [33]:
results_tactics_id = match_items(tactics_id_list, news_sources)
md_search_tacticid_df = results_to_df(results_tactics_id)
md_search_tacticid_df = md_search_tacticid_df.reset_index(drop=True)
md_search_tacticid_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


#####  **2.1.1 Añadimos la información en caso de encontrar referencias en el documento**

In [34]:
if add_information:
    # Parámetros locales
    source_rel_tactic = 'techniques_tactics' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_tactic = 'tactics' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_tacticid_df.shape[0]):
        file_path = md_search_tacticid_df.loc[new, 'file_path']
        matches = md_search_tacticid_df.loc[new, 'match_count']
        items = md_search_tacticid_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_tactic, source_info_tactic)
            data_to_incoporate['tactic_ID'] = data_to_incoporate['tactic_ID'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo tactic id
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['tactic_ID']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_tacticid_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Se han modificado un total de 0 archivo/s.


#### **2.2 Obtenemos tabla de información encontrada por nombre de táctica**

In [35]:
results_tactics_name = match_items(tactics_name_list, news_sources)
md_search_tactic_df = results_to_df(results_tactics_name)
md_search_tactic_df = md_search_tactic_df.reset_index(drop=True)
md_search_tactic_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,6,"[EXECUTION, INITIAL ACCESS, DISCOVERY, EXFILTR..."
1,c:\Users\jelopez\Documents\CyberProof\python\d...,5,"[IMPACT, EXECUTION, DISCOVERY, LATERAL MOVEMEN..."
2,c:\Users\jelopez\Documents\CyberProof\python\d...,5,"[COMMAND AND CONTROL, EXECUTION, EXFILTRATION,..."


#####  **2.2.1 Añadimos la información en caso de encontrar referencias en el documento**

In [36]:
if add_information:
    # Parámetros locales
    source_rel_tactic = 'techniques_tactics' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_tactic = 'tactics' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_tactic_df.shape[0]):
        file_path = md_search_tactic_df.loc[new, 'file_path']
        matches = md_search_tactic_df.loc[new, 'match_count']
        items = md_search_tactic_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_tactic, source_info_tactic)
            data_to_incoporate['tactic'] = data_to_incoporate['tactic'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo tactic id
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['tactic']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_tactic_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240220\cyble_chronicles__january_5_latest_findings__recommendations_for_the_cybersecurity_community_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240220\new_go-based_jkwerlo_ransomware_poses_a_risk_to_french_and_spanish_users_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240626\warzonerat_returns_with_multi-stage_attack_post_fbi_seizure_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240611\vietnamese_entities_targeted_by_china-linked_mustang_panda_in_cyber_espionage_-_cyble.md
Fichero generado y guardado correctamente c:\Use

### **3. Búsqueda de data sources (ID y nombre)**

#### **3.1 Obtenemos tabla de información encontrada por ID**

In [37]:
results_datasources_id = match_items(datasources_id_list, news_sources)
md_search_datasourcesid_df = results_to_df(results_datasources_id)
md_search_datasourcesid_df = md_search_datasourcesid_df.reset_index(drop=True)
md_search_datasourcesid_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


#####  **3.1.1 Añadimos la información en caso de encontrar referencias en el documento**

In [38]:
if add_information:
    # Parámetros locales
    source_rel_datasource = 'techniques_datasources' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_datasource = 'datasources' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_datasourcesid_df.shape[0]):
        file_path = md_search_datasourcesid_df.loc[new, 'file_path']
        matches = md_search_datasourcesid_df.loc[new, 'match_count']
        items = md_search_datasourcesid_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_datasource, source_info_datasource)
            data_to_incoporate['data_source_ID'] = data_to_incoporate['data_source_ID'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo data source id
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['data_source_ID']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_datasourcesid_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Se han modificado un total de 0 archivo/s.


In [39]:
print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Se han modificado un total de 0 archivo/s.


#### **3.2 Obtenemos tabla de información encontrada por nombre de data source**

In [40]:
results_datasources_name = match_items(datasources_name_list, news_sources)
md_search_datasources_df = results_to_df(results_datasources_name)
md_search_datasources_df = md_search_datasources_df.reset_index(drop=True)
md_search_datasources_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,13,"[USER ACCOUNT, PROCESS, SERVICE, KERNEL, PERSO..."
1,c:\Users\jelopez\Documents\CyberProof\python\d...,10,"[PROCESS, SERVICE, ACTIVE DIRECTORY, PERSONA, ..."
2,c:\Users\jelopez\Documents\CyberProof\python\d...,10,"[KERNEL, PROCESS, SERVICE, MODULE, INSTANCE, C..."


#####  **3.2.1 Añadimos la información en caso de encontrar referencias en el documento**

In [41]:
if add_information:
    # Parámetros locales
    source_rel_datasource = 'techniques_datasources' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_datasource = 'datasources' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_datasources_df.shape[0]):
        file_path = md_search_datasources_df.loc[new, 'file_path']
        matches = md_search_datasources_df.loc[new, 'match_count']
        items = md_search_datasources_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_datasource, source_info_datasource)
            data_to_incoporate['data_source'] = data_to_incoporate['data_source'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo data source
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['data_source']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_datasources_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240717\new_malware_campaign_abusing_rdpwrapper_and_tailscale_to_target_cryptocurrency_users_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240220\cyble_chronicles__january_5_latest_findings__recommendations_for_the_cybersecurity_community_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240220\sneaky_azorult_back_in_action_and_goes_undetected_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240520\in_the_shadow_of_venus_trinity_ransomwares_covert_ties_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\

### **4. Búsqueda de plataforma (nombre)**

In [42]:
results_platforms_name = match_items(platforms_name_list, news_sources)
md_search_platform_df = results_to_df(results_platforms_name)
md_search_platform_df = md_search_platform_df.reset_index(drop=True)
md_search_platform_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,4,"[WINDOWS, PRE, LINUX, NETWORK]"
1,c:\Users\jelopez\Documents\CyberProof\python\d...,4,"[WINDOWS, PRE, LINUX, NETWORK]"
2,c:\Users\jelopez\Documents\CyberProof\python\d...,3,"[WINDOWS, PRE, NETWORK]"


In [43]:
if add_information:
    # Parámetros locales
    source_rel_platform = 'techniques_platforms' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_platform = 'platforms' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_platform_df.shape[0]):
        file_path = md_search_platform_df.loc[new, 'file_path']
        matches = md_search_platform_df.loc[new, 'match_count']
        items = md_search_platform_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_platform, source_info_platform)
            data_to_incoporate['platform'] = data_to_incoporate['platform'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo platform
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['platform']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_platform_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240614\cve-2024-4577_ongoing_exploitation_of_a_critical_php_vulnerability_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240524\the_rust_revolution_new_embargo_ransomware_steps_in_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240220\cyber_espionage_attack_on_the_indian_air_force_go-based_infostealer_exploits_slack_for_data_theft_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240223\uncovering_atomic_stealer_amos_strikes_and_the_rise_of_dead_cookies_restoration_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelo

### **5. Búsqueda de software (ID y nombre)**

#### **5.1 Obtenemos tabla de información encontrada por ID**

In [44]:
results_software_id = match_items(software_id_list, news_sources)
md_search_softwareid_df = results_to_df(results_software_id)
md_search_softwareid_df = md_search_softwareid_df.reset_index(drop=True)
md_search_softwareid_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


#####  **5.1.1 Añadimos la información en caso de encontrar referencias en el documento**

In [45]:
if add_information:
    # Parámetros locales
    source_rel_software = 'techniques_software' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_software = 'software' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_softwareid_df.shape[0]):
        file_path = md_search_softwareid_df.loc[new, 'file_path']
        matches = md_search_softwareid_df.loc[new, 'match_count']
        items = md_search_softwareid_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_software, source_info_software)
            data_to_incoporate['software_ID'] = data_to_incoporate['software_ID'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo software_ID
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['software_ID']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_softwareid_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Se han modificado un total de 0 archivo/s.


#### **5.2 Obtenemos tabla de información encontrada por nombre de software**

In [46]:
results_software_name = match_items(software_name_list, news_sources)
md_search_software_df = results_to_df(results_software_name)
md_search_software_df = md_search_software_df.reset_index(drop=True)
md_search_software_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,13,"[CONTI, AZORULT, CMD, REG, NET, SYSTEMINFO, SC..."
1,c:\Users\jelopez\Documents\CyberProof\python\d...,11,"[CONTI, RTM, NET, RUBEUS, DISCO, PING, PS1, TO..."
2,c:\Users\jelopez\Documents\CyberProof\python\d...,10,"[CONTI, REG, NET, EPIC, FORFILES, TOR, PS1, AT..."


#####  **5.2.1 Añadimos la información en caso de encontrar referencias en el documento**

In [47]:
if add_information:
    # Parámetros locales
    source_rel_software = 'techniques_software' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_software = 'software' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_software_df.shape[0]):
        file_path = md_search_software_df.loc[new, 'file_path']
        matches = md_search_software_df.loc[new, 'match_count']
        items = md_search_software_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_software, source_info_software)
            data_to_incoporate['software'] = data_to_incoporate['software'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo software
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['software']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_software_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240220\sneaky_azorult_back_in_action_and_goes_undetected_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240220\new_go-based_jkwerlo_ransomware_poses_a_risk_to_french_and_spanish_users_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240611\vietnamese_entities_targeted_by_china-linked_mustang_panda_in_cyber_espionage_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240717\investigating_the_new_jellyfish_loader_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_informati

### **6. Búsqueda de grupo (ID y nombre)**

#### **6.1 Obtenemos tabla de información encontrada por ID**

In [48]:
results_group_id = match_items(groups_id_list, news_sources)
md_search_groupid_df = results_to_df(results_group_id)
md_search_groupid_df = md_search_groupid_df.reset_index(drop=True)
md_search_groupid_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


#####  **6.1.1 Añadimos la información en caso de encontrar referencias en el documento**

In [49]:
if add_information:
    # Parámetros locales
    source_rel_groups = 'techniques_groups' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_groups = 'groups' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_groupid_df.shape[0]):
        file_path = md_search_groupid_df.loc[new, 'file_path']
        matches = md_search_groupid_df.loc[new, 'match_count']
        items = md_search_groupid_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_groups, source_info_groups)
            data_to_incoporate['group_ID'] = data_to_incoporate['group_ID'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo group_ID
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['group_ID']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_groupid_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Se han modificado un total de 0 archivo/s.


#### **6.2 Obtenemos tabla de información encontrada por nombre de grupo**

In [50]:
results_group_name = match_items(groups_name_list, news_sources)
md_search_group_df = results_to_df(results_group_name)
md_search_group_df = md_search_group_df.reset_index(drop=True)
md_search_group_df.head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,4,"[APT3, TRANSPARENT TRIBE, SIDEWINDER, SIDECOPY]"
1,c:\Users\jelopez\Documents\CyberProof\python\d...,1,[MUDDYWATER]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,1,[INCEPTION]


#####  **6.2.1 Añadimos la información en caso de encontrar referencias en el documento**

In [51]:
if add_information:
    # Parámetros locales
    source_rel_groups = 'techniques_groups' # techniques_datasources / techniques_groups / techniques_platforms / techniques_software / techniques_tactics 
    source_info_groups = 'groups' # datasources / groups / platforms / software / tactics
    modified_files = []
    for new in range(md_search_group_df.shape[0]):
        file_path = md_search_group_df.loc[new, 'file_path']
        matches = md_search_group_df.loc[new, 'match_count']
        items = md_search_group_df.loc[new, 'matches']
        if matches > 0:
            data_to_incoporate = get_data_from_relation(matrix, source_rel_groups, source_info_groups)
            data_to_incoporate['group'] = data_to_incoporate['group'].apply(lambda x: str(x[0]).upper() if len(x) == 1 else x.upper()) # Quitamos lista y pasamos a mayúsculas el campo group
            header_df = pd.DataFrame(columns=data_to_incoporate.columns)
            for item in items:
                row_to_incorporate = data_to_incoporate[data_to_incoporate['group']==item] # Mapeamos el campo a buscar
                header_df = pd.concat([header_df, row_to_incorporate], ignore_index=True)
                header_df = header_df.map(list_to_string)
                for column in header_df.columns:
                    if '_description' in column:
                        header_df[column] = header_df[column].apply(clean_description_col)
                    elif 'matrix_domains' in column:
                        header_df['matrix_domains'] = header_df['matrix_domains'].apply(clean_col)

                # 1. Construimos el apéndice que contiene la información relacionada
                app = ""
                for index, row in header_df.iterrows():
                    row_number = index + 1  # Ajustar el número de fila para que comience en 1
                    for col in header_df.columns:
                        data = row[col]
                        app += f"AI_{col}_{row_number}: \"{data}\"\n"
            # 2. Obtenemos el header original
            with open(md_search_group_df.loc[new, 'file_path'], 'r', encoding='utf-8') as file:
                md_content = file.read()
                file.close()
                original_header = re.findall(r'---(.*?)---', md_content, flags=re.DOTALL)
                original_header = original_header[0]

                # 3. Reconstruimos las propiedades del .md
                modified_content = re.sub(r'---(.*?)---', f'---{original_header}\n{app}\n---', md_content, flags=re.DOTALL)

                # 4. Guardado
                base_path = os.path.join(os.path.dirname(file_path),different_output_folder)
                if not os.path.exists(base_path):
                    os.makedirs(base_path)
                save_path = os.path.join(base_path, os.path.basename(file_path))
                with open(save_path, 'w') as file:
                        file.write(modified_content)
                file.close()
                modified_files.append(save_path)
                print(f'Fichero generado y guardado correctamente {save_path}')
    print(f'Se han modificado un total de {len(modified_files)} archivo/s.')

Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240716\the_overlapping_cyber_strategies_of_transparent_tribe_and_sidecopy_against_india_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240220\cyber_espionage_attack_on_the_indian_air_force_go-based_infostealer_exploits_slack_for_data_theft_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240405\asukastealer_a_revamped_version_of_the_observerstealer_advertised_as_malware-as-a-service_-_cyble.md
Fichero generado y guardado correctamente c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\cyble\save_by_date\20240716\tiny_backdoor_goes_undetected__suspected_turla_leveraging_msbuild_to_evade_detection_-_cyble.m